# Issue #11: parking patterns and prediction benchmarks

This notebook uses Dylan's exported dataset without manual edits. Run all cells from the repository root or `notebooks/`. Select the project's `.venv` Python kernel in VS Code. If the CSV is missing, run `python -m src.build_dataset` first.

We explore historical readings, then compare training-only historical averages with additive linear regression on the same later survey dates. Findings describe the surveyed beginning-of-quarter periods, not live or full-quarter availability.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Run from the repository root or notebooks directory")
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from src.parking_benchmarks import load_analysis_dataset, FEATURES
from src.parking_analysis import export_analysis, plot_patterns

DATASET = ROOT / "data/processed/pipeline/analysis_dataset.csv"
OUTPUT = ROOT / "results/issue11"
records = load_analysis_dataset(DATASET)
print(f"Python: {sys.executable}")
print(f"{len(records):,} readings; {records.facility.nunique()} facilities")
display(records.head())

## 1. Coverage, columns and flags

Each row is one parking group at one survey time, not one space. Raw labels and provenance remain available. Missing subtype/level is unspecified, while populated restrictions stay distinct. Read the pipeline's coverage and exclusion summary before interpreting patterns; the analysis does not silently remove additional records.

In [ ]:
display(pd.DataFrame({"dtype": records.dtypes.astype(str), "missing": records.isna().sum()}))
display(Markdown((ROOT / "data/processed/pipeline/summary.md").read_text(encoding="utf-8")))

## 2. Reproduce EDA and benchmarks

The latest 20% of distinct survey dates (rounded up) forms the test set; every reading from a date stays together. Both methods use exactly the same test rows. No observed open-space count, capacity, quality flag or source identifier enters the feature set. EDA across all dates is descriptive and does not select or tune the model.

Historical means first match facility/category/subtype/level + weekday/time/quarter, then back off to parking group + time, parking group alone, and the training-wide mean. Regression uses one-hot categorical indicators and ordinary least squares. All fitting happens on training data only.

In [ ]:
result, tables = export_analysis(records, OUTPUT)
print("Model inputs:", FEATURES)
for name in ["train", "test"]:
    frame = result[name]
    print(name, len(frame), "readings;", frame.facility.nunique(), "facilities;",
          frame.observation_date.min().date(), "to", frame.observation_date.max().date())
display(result["metrics"].round(3))

## 3. Availability patterns

Means below weight each reading equally, rather than weighting by capacity. The facility and category chart uses the most-observed groups for readability; CSV tables contain all groups. Aggregate quarter changes can reflect changes in facility/category/time composition and should not be interpreted as causal demand changes.

In [ ]:
for name in ["time", "weekday", "period", "category"]:
    print(name)
    display(tables[name])
figures = plot_patterns(tables)
for figure in figures.values():
    display(figure)
    plt.close(figure)

## 4. Sparse histories and variation

The fewer-than-100-readings flag is a descriptive review threshold, not an exclusion rule. Standard deviation summarizes variability across all times/categories/periods; a high value does not alone prove inconsistent source data. Review thin coverage and changing group composition with Dylan before drawing facility-level conclusions.

In [ ]:
display(tables["coverage"].head(15))
display(tables["coverage"].sort_values("std_available", ascending=False).head(10))
for name in ["quality_flag", "label_status", "capacity_source"]:
    print(name)
    display(tables[name])

## 5. Fair comparison and near-full errors

Primary predictions are clipped to [0, 1]. Raw regression MAE and out-of-range counts/rates remain reported. MAE is expressed in percentage points. Near full means the observed fraction available is at most 10%; this definition is fixed before evaluating the models. Near-full overprediction measures optimistic error, averaging zero when a prediction does not overstate availability.

Fallback counts show how often the baseline has to pool histories. Unknown regression categories use zero indicators; neither method can infer a new facility's specific behavior from its name.

In [ ]:
display(result["metrics"].round(3))
display(result["predictions"].baseline_group.value_counts().rename("test readings"))
display(pd.read_csv(OUTPUT / "facility_errors.csv").sort_values("regression_mae_pp", ascending=False).head(10))
display(pd.read_csv(OUTPUT / "example_predictions.csv"))

## 6. Findings, limits, and next steps

The report below records the actual scores, periods, counts, exploratory findings and example predictions from this run. Discuss dataset semantics and assumptions with Dylan; that coordination is still pending. Investigate common-group quarter comparisons, date-based rolling validation, and near-full errors before adding model complexity.

In [ ]:
display(Markdown((OUTPUT / "summary.md").read_text(encoding="utf-8")))